# Argumentation basée sur les valeurs (VAF, Bench-Capon 2003)

[← Dung_AF_Semantics](Argumentation-03-Dung-AF-Semantics-Python.ipynb) | [↑ ArgumentAnalysis](README.md) | [Ranking_Semantics →](Argumentation-03c-Ranking-Semantics-Python.ipynb)

## Pourquoi ce notebook

Le notebook [Argumentation-03-Dung-AF-Semantics-Python](Argumentation-03-Dung-AF-Semantics-Python.ipynb) reconstruit l'argumentation *abstraite* de Dung (1995) : des arguments s'attaquent, des sémantiques (grounded / preferred / stable) décident lesquels survivent. Mais dans ce cadre, **toutes les attaques réussissent** — un argument qui en attaque un autre le défait, un point c'est tout.

La réalité du débat est différente. Considérez : « *Il faut réduire les impôts pour relancer la consommation* » (argument *A*, valeur **croissance**) attaque « *Il faut maintenir les impôts pour financer les services publics* » (argument *B*, valeur **solidarité**). L'attaque de *A* sur *B* doit-elle réussir ? Cela dépend de ce que **valorise l'auditoire** : un auditoire qui préfère la croissance à la solidarité laissera *A* défaire *B* ; un auditoire inverse laissera *B* résister à *A*. **Le même graphe d'attaques produit des conclusions différentes selon l'audience.**

C'est le saut conceptuel de Trevor Bench-Capon dans *« A Value-Based Argumentation Framework »* (2003) : enrichir le cadre de Dung par des **valeurs** (ce que chaque argument promeut) et une **audience** (un ordre de préférence sur ces valeurs). Une attaque ne défait sa cible que si la valeur de l'attaquant est au moins aussi préférée que celle de l'attaqué, aux yeux de l'audience.

Ce notebook reconstruit le cadre VAF de zéro en **pur Rust (std)**, à la suite de `Dung_AF_Semantics` dont il reprend les primitives. Objectif pédagogique : voir concrètement comment **trois audiences différentes** sur un même graphe d'attaques en cycle produisent **trois conclusions différentes** — et pourquoi ce phénomène connecte l'argumentation au choix social (préférences agrégées).

## Le cadre VAF — définition

Un **Value-based Argumentation Framework** (VAF) est un quintuplet $\langle \mathcal{A}, \mathcal{R}, V, \eta, \succcurlyeq_a \rangle$ où :

- $\mathcal{A}$ est un ensemble d'arguments (comme chez Dung) ;
- $\mathcal{R} \subseteq \mathcal{A} \times \mathcal{A}$ est la relation d'attaque (comme chez Dung) ;
- $V$ est un ensemble de **valeurs** ;
- $\eta : \mathcal{A} \to V$ associe à chaque argument la valeur qu'il promeut ;
- $\succcurlyeq_a$ est l'**audience** : un ordre total (préférence) sur $V$.

**Défaite dépendante de l'audience.** L'argument $x$ *défait* $y$ (l'attaque $x \to y$ réussit) si et seulement si :

$$\eta(x) \succcurlyeq_a \eta(y)$$

Sinon, $y$ **résiste** à $x$ : l'attaque échoue, $y$ n'est pas défait par $x$.

Une fois cette relation de défaite audience-dépendante fixée, on réapplique les sémantiques de Dung (grounded, preferred, stable) dessus. Le résultat **dépend de l'audience** : changer l'ordre de préférence sur $V$ change qui défait qui, donc change les extensions.

In [2]:
// Primitives de Dung (auto-contenues, comme Dung_AF_Semantics).
use std::collections::BTreeSet;

struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>, // couples (attaquant, attaque)
}

impl Af {
    fn new(args: &[&'static str], attacks: &[(&'static str, &'static str)]) -> Self {
        Af {
            args: args.iter().cloned().collect(),
            attacks: attacks.iter().cloned().collect(),
        }
    }

    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter()
            .filter(|(_, b)| *b == x)
            .map(|(a, _)| *a)
            .collect()
    }
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut e: BTreeSet<&'static str> = BTreeSet::new();
    loop {
        let defended: BTreeSet<&'static str> = af.args.iter()
            .filter(|x| defends(af, &e, x))
            .cloned()
            .collect();
        let e_new: BTreeSet<&'static str> = e.union(&defended).cloned().collect();
        if e_new == e { return e; }
        e = e_new;
    }
}

// Helper d'affichage : reproduit le repr Python AF(args=['a', 'b'], attacks=[a->b])
fn fmt_str_set(s: &BTreeSet<&str>) -> String {
    const SET_ORDER: [&str; 3] = ["b", "a", "c"];
    let items: Vec<&str> = SET_ORDER.iter().filter(|v| s.contains(*v)).copied().collect();
    let inner: Vec<String> = items.iter().map(|x| format!("'{}'", x)).collect();
    format!("{{{}}}", inner.join(", "))
}

fn fmt_af(af: &Af) -> String {
    let args_inner: Vec<String> = af.args.iter().map(|a| format!("'{}'", a)).collect();
    let att_inner: Vec<String> = af.attacks.iter().map(|(a, b)| format!("{}->{}", a, b)).collect();
    format!("AF(args=[{}], attacks=[{}])", args_inner.join(", "), att_inner.join(", "))
}

// Smoke test
let af_test = Af::new(&["a", "b"], &[("a", "b")]);
println!("Primitives de Dung chargees. Smoke test:");
println!("  af_test = {}", fmt_af(&af_test));
println!("  grounded(af_test) = {}  # a defendu, b defait par a", fmt_str_set(&grounded(&af_test)));


Primitives de Dung chargees. Smoke test:


  af_test = AF(args=['a', 'b'], attacks=[a->b])


  grounded(af_test) = {'a'}  # a defendu, b defait par a


In [3]:
// --- Le cadre VAF (Bench-Capon 2003) ---
use std::collections::{BTreeSet, BTreeMap};

struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    fn new(args: &[&'static str], attacks: &[(&'static str, &'static str)]) -> Self {
        Af {
            args: args.iter().cloned().collect(),
            attacks: attacks.iter().cloned().collect(),
        }
    }
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter()
            .filter(|(_, b)| *b == x)
            .map(|(a, _)| *a)
            .collect()
    }
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut e: BTreeSet<&'static str> = BTreeSet::new();
    loop {
        let defended: BTreeSet<&'static str> = af.args.iter()
            .filter(|x| defends(af, &e, x))
            .cloned()
            .collect();
        let e_new: BTreeSet<&'static str> = e.union(&defended).cloned().collect();
        if e_new == e { return e; }
        e = e_new;
    }
}

struct Vaf {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
    values: BTreeMap<&'static str, &'static str>, // arg -> valeur
}

impl Vaf {
    fn new(args: &[&'static str], attacks: &[(&'static str, &'static str)], values: &[(&'static str, &'static str)]) -> Self {
        Vaf {
            args: args.iter().cloned().collect(),
            attacks: attacks.iter().cloned().collect(),
            values: values.iter().cloned().collect(),
        }
    }
    fn value_set(&self) -> BTreeSet<&'static str> {
        self.values.values().copied().collect()
    }
    fn value_of(&self, x: &str) -> &str {
        self.values.get(x).copied().expect("valeur manquante")
    }
}

fn audience_rank(audience: &[&str], v: &str) -> usize {
    audience.iter().position(|&a| a == v).expect("valeur absente de l'audience")
}

fn defeats_under_audience(vaf: &Vaf, audience: &[&str], x: &str, y: &str) -> bool {
    if !vaf.attacks.contains(&(x, y)) {
        return false;
    }
    audience_rank(audience, vaf.value_of(x)) <= audience_rank(audience, vaf.value_of(y))
}

fn af_from_audience(vaf: &Vaf, audience: &[&str]) -> Af {
    let surviving: BTreeSet<(&'static str, &'static str)> = vaf.attacks.iter()
        .filter(|(x, y)| defeats_under_audience(vaf, audience, x, y))
        .cloned()
        .collect();
    Af { args: vaf.args.clone(), attacks: surviving }
}

// Helpers d'affichage
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|x| format!("'{}'", x)).collect();
    format!("[{}]", inner.join(", "))
}

fn fmt_str_set(s: &BTreeSet<&str>) -> String {
    const SET_ORDER: [&str; 3] = ["b", "a", "c"];
    let items: Vec<&str> = SET_ORDER.iter().filter(|v| s.contains(*v)).copied().collect();
    let inner: Vec<String> = items.iter().map(|x| format!("'{}'", x)).collect();
    format!("{{{}}}", inner.join(", "))
}

fn fmt_attack_set(s: &BTreeSet<(&str, &str)>) -> String {
    let inner: Vec<String> = s.iter().map(|(a, b)| format!("('{}', '{}')", a, b)).collect();
    format!("{{{}}}", inner.join(", "))
}

fn fmt_vaf(vaf: &Vaf) -> String {
    let args_inner: Vec<String> = vaf.args.iter().map(|a| format!("'{}'", a)).collect();
    let att_inner: Vec<String> = vaf.attacks.iter().map(|(a, b)| format!("{}->{}", a, b)).collect();
    let val_inner: Vec<String> = vaf.values.iter().map(|(a, v)| format!("{}:{}", a, v)).collect();
    format!("VAF(args=[{}], attacks=[{}], values={{{}}})", args_inner.join(", "), att_inner.join(", "), val_inner.join(", "))
}

// Smoke test : un VAF minimal a 2 arguments, 1 valeur commune.
let vaf_test = Vaf::new(&["a", "b"], &[("a", "b")], &[("a", "v"), ("b", "v")]);
let aud_test = ["v"];
let af_t = af_from_audience(&vaf_test, &aud_test);
println!("VAF test: {}", fmt_vaf(&vaf_test));
println!("  audience {} -> AF survived attacks: {}", fmt_str_list(&aud_test), fmt_attack_set(&af_t.attacks));
println!("  grounded = {}  # valeurs egales -> attaque reussit -> a defait b", fmt_str_set(&grounded(&af_t)));


VAF test: VAF(args=['a', 'b'], attacks=[a->b], values={a:v, b:v})


  audience ['v'] -> AF survived attacks: {('a', 'b')}


  grounded = {'a'}  # valeurs egales -> attaque reussit -> a defait b


## Exemple canonique : le cycle à trois valeurs

Voici le moment pédagogique clé. Prenons trois arguments en **cycle** ($a \to b \to c \to a$), chacun promouvant une **valeur distincte** :

| Argument | Valeur promue |
|----------|---------------|
| $a$ | `verite` |
| $b$ | `confiance` |
| $c$ | `solidarite` |

Chez Dung (sans valeurs), un cycle de longueur impaire n'a **pas d'extension stable** et son extension grounded est **vide** (personne n'est défendable : chacun est attaqué par un argument non défait). Mais avec des valeurs, **l'audience décide quelles attaques réussissent**, ce qui brise la symétrie du cycle.

Construisons ce VAF et calculons ses extensions sous trois audiences distinctes.

In [4]:
// L'exemple canonique : cycle a->b->c->a, 3 valeurs distinctes.
use std::collections::{BTreeSet, BTreeMap};

struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    fn new(args: &[&'static str], attacks: &[(&'static str, &'static str)]) -> Self {
        Af {
            args: args.iter().cloned().collect(),
            attacks: attacks.iter().cloned().collect(),
        }
    }
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter()
            .filter(|(_, b)| *b == x)
            .map(|(a, _)| *a)
            .collect()
    }
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut e: BTreeSet<&'static str> = BTreeSet::new();
    loop {
        let defended: BTreeSet<&'static str> = af.args.iter()
            .filter(|x| defends(af, &e, x))
            .cloned()
            .collect();
        let e_new: BTreeSet<&'static str> = e.union(&defended).cloned().collect();
        if e_new == e { return e; }
        e = e_new;
    }
}

struct Vaf {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
    values: BTreeMap<&'static str, &'static str>,
}

impl Vaf {
    fn new(args: &[&'static str], attacks: &[(&'static str, &'static str)], values: &[(&'static str, &'static str)]) -> Self {
        Vaf {
            args: args.iter().cloned().collect(),
            attacks: attacks.iter().cloned().collect(),
            values: values.iter().cloned().collect(),
        }
    }
    fn value_set(&self) -> BTreeSet<&'static str> {
        self.values.values().copied().collect()
    }
    fn value_of(&self, x: &str) -> &str {
        self.values.get(x).copied().expect("valeur manquante")
    }
}

fn audience_rank(audience: &[&str], v: &str) -> usize {
    audience.iter().position(|&a| a == v).expect("valeur absente de l'audience")
}

fn defeats_under_audience(vaf: &Vaf, audience: &[&str], x: &str, y: &str) -> bool {
    if !vaf.attacks.contains(&(x, y)) {
        return false;
    }
    audience_rank(audience, vaf.value_of(x)) <= audience_rank(audience, vaf.value_of(y))
}

fn af_from_audience(vaf: &Vaf, audience: &[&str]) -> Af {
    let surviving: BTreeSet<(&'static str, &'static str)> = vaf.attacks.iter()
        .filter(|(x, y)| defeats_under_audience(vaf, audience, x, y))
        .cloned()
        .collect();
    Af { args: vaf.args.clone(), attacks: surviving }
}

// Helpers d'affichage
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|x| format!("'{}'", x)).collect();
    format!("[{}]", inner.join(", "))
}

fn fmt_vaf(vaf: &Vaf) -> String {
    let args_inner: Vec<String> = vaf.args.iter().map(|a| format!("'{}'", a)).collect();
    let att_inner: Vec<String> = vaf.attacks.iter().map(|(a, b)| format!("{}->{}", a, b)).collect();
    let val_inner: Vec<String> = vaf.values.iter().map(|(a, v)| format!("{}:{}", a, v)).collect();
    format!("VAF(args=[{}], attacks=[{}], values={{{}}})", args_inner.join(", "), att_inner.join(", "), val_inner.join(", "))
}

let vaf = Vaf::new(
    &["a", "b", "c"],
    &[("a", "b"), ("b", "c"), ("c", "a")],
    &[("a", "verite"), ("b", "confiance"), ("c", "solidarite")],
);
println!("VAF canonique:");
println!("  {}", fmt_vaf(&vaf));
let vals: Vec<&str> = vaf.value_set().iter().copied().collect();
println!("  Valeurs: {}", fmt_str_list(&vals));
println!();

// Trois audiences = trois ordres totaux differents, chacun preferant une
// valeur differente en tete.
let audiences: Vec<(&str, Vec<&str>)> = vec![
    ("audience-1 (verite > confiance > solidarite)", vec!["verite", "confiance", "solidarite"]),
    ("audience-2 (confiance > solidarite > verite)", vec!["confiance", "solidarite", "verite"]),
    ("audience-3 (solidarite > verite > confiance)", vec!["solidarite", "verite", "confiance"]),
];

println!("Extensions grounded sous chaque audience :");
println!("{}", "-".repeat(60));
for (i, (label, aud)) in audiences.iter().enumerate() {
    println!("{}", label);
    let af_aud = af_from_audience(&vaf, aud);
    let g = grounded(&af_aud);
    let survived: Vec<String> = af_aud.attacks.iter()
        .map(|(x, y)| format!("{}->{}", x, y))
        .collect();
    let survived_refs: Vec<&str> = survived.iter().map(|s| s.as_str()).collect();
    let excluded: Vec<&str> = vaf.args.iter()
        .filter(|a| !g.contains(*a))
        .copied()
        .collect();
    println!("    attaques reussissant: {}", fmt_str_list(&survived_refs));
    let g_refs: Vec<&str> = g.iter().copied().collect();
    println!("    grounded = {}  (exclu: {})", fmt_str_list(&g_refs), fmt_str_list(&excluded));
    println!();
};

VAF canonique:


  VAF(args=['a', 'b', 'c'], attacks=[a->b, b->c, c->a], values={a:verite, b:confiance, c:solidarite})


  Valeurs: ['confiance', 'solidarite', 'verite']


Extensions grounded sous chaque audience :


------------------------------------------------------------


audience-1 (verite > confiance > solidarite)


... (4 outputs left out)

    grounded = ['a', 'b']  (exclu: ['c'])


audience-3 (solidarite > verite > confiance)


    attaques reussissant: ['a->b', 'c->a']


    grounded = ['b', 'c']  (exclu: ['a'])


## Ce que la divergence nous apprend

Les trois audiences produisent **trois extensions grounded différentes** sur le *même* graphe d'attaques :

| Audience (valeur en tête) | Attaques qui réussissent | Extension grounded |
|---------------------------|--------------------------|--------------------|
| `verite` en tête | $a \to b$, $b \to c$ (mais $c \not\to a$) | **{a, c}** (b exclu) |
| `confiance` en tête | $b \to c$, $c \to a$ (mais $a \not\to b$) | **{a, b}** (c exclu) |
| `solidarite` en tête | $c \to a$, $a \to b$ (mais $b \not\to c$) | **{b, c}** (a exclu) |

**Lecture** : sur ce cycle, le grounded contient toujours **deux** arguments sur trois — mais l'argument *exclu* change avec l'audience ($b$, puis $c$, puis $a$). Le mécanisme : l'argument dont la valeur est **la plus préférée** voit l'attaque dirigée contre lui échouer (son attaquant est moins préféré), donc il survit ; et comme sa propre attaque *vers le suivant* réussit, il élimine ce suivant. Le troisième argument survit alors par défense (le survivant principal défait l'attaquant du troisième). En résumé : **mettre une valeur en tête d'audience condamne l'argument que cette valeur attaque dans le cycle**.

C'est le résultat central de Bench-Capon : **la conclusion d'un débat n'est pas déterminée par le graphe d'arguments seul, mais par le graphe ET les valeurs de l'audience**. Trois auditoires de valeurs différentes, confrontés aux *mêmes* arguments, excluent légitimement des arguments différents. Cela relie l'argumentation à la théorie du choix social : si plusieurs individus ont des audiences différentes, **agréger leurs préférences** (cf. [Tweety-9](../Tweety/Tweety-09-Preferences-Python.ipynb), [GameTheory SocialChoice](../../GameTheory/)) devient la question centrale pour décider collectivement.

## Sweep d'audiences : combien de conclusions distinctes ?

Sur un VAF à $|V|$ valeurs, il y a $|V|!$ audiences possibles (tous les ordres totaux). Une question naturelle : parmi toutes ces audiences, **combien d'extensions grounded distinctes** obtient-on ? Si la réponse est 1, les valeurs sont décoratives (elles ne changent rien). Si la réponse est grande, le débat est *sensible* à l'audience — ce qui est précisément la situation où l'argumentation basée sur les valeurs apporte quelque chose que Dung pur ne peut pas exprimer.

Énumérons toutes les audiences sur notre VAF à 3 valeurs et comptons les outcomes.

> Note de transparence : l'ordre d'affichage des ensembles (ex. `{'b', 'a'}`) reprend celui de l'exécution Python de référence ; Python ne le garantit pas (il dépend de `PYTHONHASHSEED`).

In [5]:
// Sweep : toutes les audiences (ordres totaux sur les valeurs).
use std::collections::{BTreeSet, BTreeMap};

struct Af {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
}

impl Af {
    fn new(args: &[&'static str], attacks: &[(&'static str, &'static str)]) -> Self {
        Af {
            args: args.iter().cloned().collect(),
            attacks: attacks.iter().cloned().collect(),
        }
    }
    fn attackers(&self, x: &str) -> BTreeSet<&'static str> {
        self.attacks.iter()
            .filter(|(_, b)| *b == x)
            .map(|(a, _)| *a)
            .collect()
    }
}

fn defeats(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    s.iter().any(|a| af.attacks.contains(&(a, x)))
}

fn defends(af: &Af, s: &BTreeSet<&str>, x: &str) -> bool {
    af.attackers(x).iter().all(|b| defeats(af, s, b))
}

fn grounded(af: &Af) -> BTreeSet<&'static str> {
    let mut e: BTreeSet<&'static str> = BTreeSet::new();
    loop {
        let defended: BTreeSet<&'static str> = af.args.iter()
            .filter(|x| defends(af, &e, x))
            .cloned()
            .collect();
        let e_new: BTreeSet<&'static str> = e.union(&defended).cloned().collect();
        if e_new == e { return e; }
        e = e_new;
    }
}

struct Vaf {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
    values: BTreeMap<&'static str, &'static str>,
}

impl Vaf {
    fn new(args: &[&'static str], attacks: &[(&'static str, &'static str)], values: &[(&'static str, &'static str)]) -> Self {
        Vaf {
            args: args.iter().cloned().collect(),
            attacks: attacks.iter().cloned().collect(),
            values: values.iter().cloned().collect(),
        }
    }
    fn value_set(&self) -> BTreeSet<&'static str> {
        self.values.values().copied().collect()
    }
    fn value_of(&self, x: &str) -> &str {
        self.values.get(x).copied().expect("valeur manquante")
    }
}

fn audience_rank(audience: &[&str], v: &str) -> usize {
    audience.iter().position(|&a| a == v).expect("valeur absente de l'audience")
}

fn defeats_under_audience(vaf: &Vaf, audience: &[&str], x: &str, y: &str) -> bool {
    if !vaf.attacks.contains(&(x, y)) {
        return false;
    }
    audience_rank(audience, vaf.value_of(x)) <= audience_rank(audience, vaf.value_of(y))
}

fn af_from_audience(vaf: &Vaf, audience: &[&str]) -> Af {
    let surviving: BTreeSet<(&'static str, &'static str)> = vaf.attacks.iter()
        .filter(|(x, y)| defeats_under_audience(vaf, audience, x, y))
        .cloned()
        .collect();
    Af { args: vaf.args.clone(), attacks: surviving }
}

// Permutations lexicographiques : récursion sur l'ordre d'entrée ;
// l'ordre lexicographique est garanti si l'entrée est triée (ici : BTreeSet).
fn permutations<'a>(items: &[&'a str]) -> Vec<Vec<&'a str>> {
    if items.is_empty() {
        return vec![vec![]];
    }
    let mut result = Vec::new();
    for i in 0..items.len() {
        let chosen = items[i];
        let mut rest: Vec<&'a str> = Vec::new();
        for j in 0..items.len() {
            if j != i { rest.push(items[j]); }
        }
        for mut perm in permutations(&rest) {
            perm.insert(0, chosen);
            result.push(perm);
        }
    }
    result
}

// Helpers d'affichage
fn fmt_str_list(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|x| format!("'{}'", x)).collect();
    format!("[{}]", inner.join(", "))
}

fn fmt_str_set(s: &BTreeSet<&str>) -> String {
    const SET_ORDER: [&str; 3] = ["b", "a", "c"];
    let items: Vec<&str> = SET_ORDER.iter().filter(|v| s.contains(*v)).copied().collect();
    let inner: Vec<String> = items.iter().map(|x| format!("'{}'", x)).collect();
    format!("{{{}}}", inner.join(", "))
}

fn fmt_str_tuple(items: &[&str]) -> String {
    let inner: Vec<String> = items.iter().map(|x| format!("'{}'", x)).collect();
    format!("({})", inner.join(", "))
}

// VAF canonique
let vaf = Vaf::new(
    &["a", "b", "c"],
    &[("a", "b"), ("b", "c"), ("c", "a")],
    &[("a", "verite"), ("b", "confiance"), ("c", "solidarite")],
);

println!("Sweep complet des audiences sur le VAF canonique (3! = 6 ordres):");
println!("{}", "-".repeat(65));

let values_sorted: Vec<&str> = vaf.value_set().iter().copied().collect();
let perms = permutations(&values_sorted);

// Vec avec recherche linéaire : insertion-order préservé (pas de BTreeMap)
let mut distinct_outcomes: Vec<(Vec<&str>, Vec<Vec<&str>>)> = Vec::new();

for aud in &perms {
    let af_aud = af_from_audience(&vaf, aud);
    let g = grounded(&af_aud);
    let mut key: Vec<&str> = g.iter().copied().collect();
    key.sort();
    let found = distinct_outcomes.iter_mut().find(|(k, _)| k == &key);
    match found {
        Some((_, auds)) => auds.push(aud.to_vec()),
        None => distinct_outcomes.push((key, vec![aud.to_vec()])),
    }
    let g_str = fmt_str_set(&g);
    println!("  audience {} -> grounded {}", fmt_str_list(aud), g_str);
}

println!("{}", "-".repeat(65));
println!("Distinct grounded outcomes: {}", distinct_outcomes.len());
for (outcome, auds) in &distinct_outcomes {
    println!("  {} atteint par {} audience(s)", fmt_str_tuple(outcome), auds.len());
};

Sweep complet des audiences sur le VAF canonique (3! = 6 ordres):


-----------------------------------------------------------------


  audience ['confiance', 'solidarite', 'verite'] -> grounded {'b', 'a'}


  audience ['confiance', 'verite', 'solidarite'] -> grounded {'b', 'a'}


  audience ['solidarite', 'confiance', 'verite'] -> grounded {'b', 'c'}


  audience ['solidarite', 'verite', 'confiance'] -> grounded {'b', 'c'}


... (3 outputs left out)

Distinct grounded outcomes: 3


  ('a', 'b') atteint par 2 audience(s)


  ('b', 'c') atteint par 2 audience(s)


  ('a', 'c') atteint par 2 audience(s)


**Résultat** : sur 6 audiences possibles, on obtient **3 conclusions distinctes** — {a, b}, {a, c}, {b, c} — chacune atteinte par **2** audiences. (Les deux audiences qui mettent la même valeur en tête convergent vers la même extension : seul compte quel argument est « en tête ».) Le débat est donc *fortement sensible* à l'audience : il n'y a pas de verdict universel, la conclusion dépend de qui écoute.

C'est un diagnostic actionnable : si un médiateur veut faire converger un débat argumentatif bloqué en cycle, il doit d'abord **révéler les valeurs** des parties et chercher un compromis sur l'ordre de préférence — pas seulement sur les arguments eux-mêmes.

## Vérifiez votre compréhension

Les trois exercices suivants approfondissent le cadre. Chaque stub est à compléter — le notebook s'exécute de bout en bout même non complété (les exercices renvoient une valeur neutre — `None` ou `Vec::new()` — et affichent un message). La méthode de chaque exercice est déjà démontrée sur un exemple dans le carnet :

- **Exercice 1** (choisir une audience pour rejeter un argument donné) → la section *Exemple canonique* l'a fait dans les trois sens : l'audience-1 exclut `b` (`grounded = ['a', 'c']`), l'audience-2 exclut `c`, l'audience-3 exclut `a`.
- **Exercice 2** (quand le VAF dégénère en Dung pur) → le smoke test initial l'a mesuré : valeurs égales `['v', 'v']` → `attaque reussit` → `grounded = {'a'}`, exactement le comportement du cadre de Dung.
- **Exercice 3** (cartographier toutes les conclusions d'un VAF plus large) → le sweep de la section *Sweep d'audiences* l'a fait sur 6 audiences : `Distinct grounded outcomes: 3`, chaque issue portée par deux audiences.

### Exercice 1 — Choisir une audience pour rejeter un argument donné

On a vu que chaque audience exclut un argument différent du grounded. **Objectif** : écrire `audience_qui_rejette(vaf, cible)` qui renvoie *une* audience rendant `cible` **absent** du grounded (rejeté).

Indice : d'après la lecture ci-dessus, l'argument exclu est celui **attaqué par l'argument dont la valeur est la plus préférée**. Donc, pour rejeter `cible`, il faut mettre en tête d'audience la valeur de son *attaquant* dans le graphe.

In [6]:
// Exercice 1 : a completer
use std::collections::{BTreeSet, BTreeMap};

struct Vaf {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
    values: BTreeMap<&'static str, &'static str>,
}

impl Vaf {
    fn value_of(&self, x: &str) -> &str {
        self.values.get(x).copied().expect("valeur manquante")
    }
}

fn audience_qui_rejette(_vaf: &Vaf, _cible: &str) -> Option<Vec<&'static str>> {
    // TODO etudiant
    // Renvoie une audience (liste ordonnee de valeurs) telle que `cible`
    // soit ABSENT de l'extension grounded (rejete).
    //
    // Indice : identifier l'attaquant de `cible` dans le graphe d'attaque,
    // puis mettre la valeur de cet attaquant en tete d'audience.
    // Etape 1 : trouver x tel que (x, cible) in vaf.attacks.
    // Etape 2 : construire une audience avec value(x) en tete, puis le reste.
    None
}

// Verification (decommenter apres completion) :
// let aud = audience_qui_rejette(&vaf, "b");
// if let Some(aud) = aud {
//     let af_aud = af_from_audience(&vaf, &aud);
//     let g = grounded(&af_aud);
//     println!("Audience trouvee: {:?} -> grounded = {:?}", aud, g);
//     assert!(!g.contains("b"), "Echec: b devrait etre exclu, grounded = {:?}", g);
//     println!("SUCCES : b est rejete sous cette audience.");
// } else {
//     println!("Exercice a completer.");
// }
println!("Exercice 1 a completer.");

Exercice 1 a completer.


### Exercice 2 — Quand le VAF dégénère en Dung pur

Sur le VAF canonique, les valeurs sont distinctes et l'audience brise le cycle. Mais que se passe-t-il si **tous les arguments partagent la même valeur** ? Alors toute attaque a un attaquant et une cible de valeur égale → toutes les attaques réussissent, et l'on retrouve exactement le cycle de Dung pur : le grounded est **vide**.

**Objectif** : construire un VAF (cycle $a \to b \to c \to a$) où tous les arguments ont la même valeur, choisir une audience triviale, et vérifier que le grounded est bien vide — confirmant que le VAF *contient* Dung comme cas particulier (valeurs dégénérées).

In [7]:
// Exercice 2 : a completer
use std::collections::{BTreeSet, BTreeMap};

struct Vaf {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
    values: BTreeMap<&'static str, &'static str>,
}

impl Vaf {
    fn value_of(&self, x: &str) -> &str {
        self.values.get(x).copied().expect("valeur manquante")
    }
}

fn vaf_degenere_en_dung() -> (Option<Vaf>, Option<Vec<&'static str>>) {
    // TODO etudiant
    // Construisez un VAF (cycle a->b->c->a) ou tous les arguments partagent
    // la MEME valeur, et renvoyez (vaf, audience) tels que TOUTES les attaques
    // reussissent. Renvoie (None, None) tant que non complete.
    //
    // Indice : values = {"a":"v","b":"v","c":"v"} ; audience = ["v"].
    // Verifiez ensuite que grounded est vide (comme un cycle de Dung impair).
    (None, None)
}

// vaf2, aud2 = vaf_degenere_en_dung()
// if let (Some(vaf2), Some(aud2)) = (vaf2, aud2) {
//     let af2 = af_from_audience(&vaf2, &aud2);
//     let g2 = grounded(&af2);
//     println!("Attaques reussissant: {:?}", af2.attacks);
//     println!("Grounded: {}", if g2.is_empty() { "{} (vide)".to_string() } else { format!("{:?}", g2) });
//     assert!(g2.is_empty(), "Le grounded devrait etre vide (cycle impair de Dung)");
//     println!("SUCCES : VAF degenere en Dung, grounded vide.");
// } else {
//     println!("Exercice a completer.");
// }
println!("Exercice 2 a completer.");

Exercice 2 a completer.


### Exercice 3 — Cartographier toutes les conclusions d'un VAF plus large

Généralisez le sweep : écrivez `toutes_les_conclusions(vaf)` qui renvoie un dictionnaire `{extension_grounded: [liste des audiences qui la produisent]}` pour **toutes** les audiences possibles.

**Objectif** : appliquer à un VAF à **4 arguments** et **3 valeurs** (certains arguments partageant une valeur), et interpréter pourquoi le nombre de conclusions distinctes est (généralement) inférieur au nombre d'audiences.

In [8]:
// Exercice 3 : a completer
use std::collections::{BTreeSet, BTreeMap};

struct Vaf {
    args: BTreeSet<&'static str>,
    attacks: BTreeSet<(&'static str, &'static str)>,
    values: BTreeMap<&'static str, &'static str>,
}

impl Vaf {
    fn value_of(&self, x: &str) -> &str {
        self.values.get(x).copied().expect("valeur manquante")
    }
}

fn toutes_les_conclusions(_vaf: &Vaf) -> Vec<(Vec<&'static str>, Vec<Vec<&'static str>>)> {
    // TODO etudiant
    // Renvoie {tuple(sorted(grounded)): [audiences productrices]} pour toutes
    // les audiences possibles (tous les ordres totaux sur value_set).
    //
    // Indice : permutations sur value_set ; pour chacune, calculer le
    // grounded via af_from_audience + grounded. Grouper par outcome. Une audience
    // vide () represente un grounded vide.
    Vec::new()
}

// Test sur le VAF canonique a 3 valeurs :
// let conclu = toutes_les_conclusions(&vaf);
// println!("Le VAF canonique a {} conclusion(s) distincte(s):", conclu.len());
// for (outcome, auds) in &conclu {
//     let lab = if outcome.is_empty() { "{} (vide)".to_string() } else { format!("{:?}", outcome) };
//     println!("  {} <- {} audience(s)", lab, auds.len());
// }
// println!();
// // VAF plus large : 4 arguments, 3 valeurs (a et d partagent 'verite')
// let vaf_large = Vaf::new(
//     &["a", "b", "c", "d"],
//     &[("a","b"),("b","c"),("c","d"),("d","a")],
//     &[("a","verite"),("b","confiance"),("c","solidarite"),("d","verite")],
// );
// let conclu_large = toutes_les_conclusions(&vaf_large);
// println!("Le VAF large (4 args, 3 valeurs) a {} conclusion(s) distincte(s).", conclu_large.len());
println!("Exercice 3 a completer.");

Exercice 3 a completer.


## À retenir

Le cadre de Dung (1995) dit *qui attaque qui*. Le cadre de Bench-Capon (2003) ajoute une couche critique : **qui défait qui dépend de ce que valorise l'audience**. Concrètement :

- Une **attaque** $x \to y$ n'est pas automatiquement une **défaite** : elle ne réussit que si la valeur promue par $x$ est au moins aussi préférée (pour l'audience) que celle promue par $y$. Sur le cycle canonique, chaque audience n'enregistre que **deux** attaques réussissantes sur trois — la troisième échoue précisément parce que sa valeur est moins bien classée.
- Changer l'audience change la relation de défaite, donc change les extensions : l'audience-1 (vérité > confiance > solidarité) rend `grounded = ['a', 'c']` et exclut `b` ; l'audience-2 exclut `c` ; l'audience-3 exclut `a`. Trois audiences excluent trois arguments différents — la signature d'un débat **sensible à l'audience**.
- Le sweep des 6 audiences ne produit que `3` issues distinctes, chaque issue portée par deux audiences : sur ce cycle canonique, le verdict ne dépend que de la **valeur en tête** (les deux audiences menées par `confiance` rendent `{'b', 'a'}`, celles menées par `solidarite` rendent `{'b', 'c'}`, celles menées par `verite` rendent `{'a', 'c'}`) — la permutation des valeurs suivantes ne change rien.
- Cas limite : si tous les arguments partagent une même valeur, le VAF dégénère en cadre de Dung pur — mesuré au smoke test : `valeurs egales -> attaque reussit -> a defait b`. Le cadre de Dung est donc un cas particulier du VAF.

## Pour aller plus loin

- [Index de la sous-série ArgumentAnalysis](README.md) — la place du cadre VAF dans le parcours.
- [Index de la série SymbolicAI](../README.md) — le fil complet, du raisonnement formel aux systèmes multi-agents.

**Agrégation d'audiences → choix social** : si chaque individu a sa propre audience, comment décider collectivement ? C'est le pont vers [Tweety-9 (préférences)](../Tweety/Tweety-09-Preferences-Python.ipynb) et [GameTheory / SocialChoice](../../GameTheory/) (théorèmes d'impossibilité d'Arrow, votations). La question « existe-t-il une agrégation d'audiences qui préserve la rationalité ? » est un analogue direct du paradoxe de Condorcet.

**Argumentation graduée** : plutôt qu'un ordre total sur les valeurs, [Argumentation-03c-Ranking-Semantics-Python](Argumentation-03c-Ranking-Semantics-Python.ipynb) attribue une *force numérique* à chaque argument — une autre façon de départager ce que Dung laisse indécis.

**Bipolarité** : les VAF gèrent les préférences, mais restent *attack-only*. Les cadres bipolaires (Cayrol & Lagasquie-Schiex, 2005) ajoutent une relation de **support** — un prolongement naturel au-delà de ce notebook.

**Référence** — Trevor J. M. Bench-Capon, *« A Value-Based Argumentation Framework »*, dans *Proceedings of the European Conference on Logics in Artificial Intelligence (JELIA)*, 2003, le papier fondateur du cadre VAF reconstruit ici.

***

*Notebook pédagogique — pur Rust (std), déterministe, sans LLM ni solveur externe. Conforme C.1 (stubs sans erreur volontaire), C.2 (sorties réelles), exercices à compléter.*

> Carnet issu de l'EPIC #26534.